# Dataset trafic pour une année (old_data)

Choisis l'année ci-dessous puis lance tout le notebook (**Run All**).
Il lit les fichiers `old_data/<année>/*.txt`, garde uniquement nos 3 arcs et enregistre
`data/processed/annees/trafic_<année>.csv` (même format que `trafic_3_arcs.csv`).

Compter environ 1 minute par année.

In [1]:
YEAR = "2014"

In [2]:
from pathlib import Path

import pandas as pd

OLD_DATA_DIR = Path("old_data")
OUT_DIR = Path("data/processed/annees")

# Même définition des arcs que dans data_exploration.ipynb (noeud amont, noeud aval)
ARCS_CIBLES = {
    "Champs-Elysees": ("Av_Champs_Elysees-Washington", "Av_Champs_Elysees-Berri"),
    "Convention": ("Lecourbe-Convention", "Convention-Blomet"),
    "Saints-Peres": ("Sts_Peres-Voltaire", "Sts_Peres-Universite"),
}

# Colonnes utiles de l'ancien format (on ignore 'dessin' = la géométrie, très lourde)
COLONNES = ["iu_ac", "libelle_nd_amont", "libelle_nd_aval", "t_1h", "q", "k", "etat_trafic", "etat_barre"]

# Dans l'ancien format, l'état du trafic est un code
ETAT_TRAFIC = {0: "Inconnu", 1: "Fluide", 2: "Pré-saturé", 3: "Saturé", 4: "Bloqué"}

## 1. Lecture des fichiers de l'année (filtrés sur nos 3 arcs)

In [3]:
fichiers = sorted((OLD_DATA_DIR / YEAR).glob("*.txt"))
print(f"{len(fichiers)} fichiers pour {YEAR}")

morceaux = []
for i, f in enumerate(fichiers, 1):
    brut = pd.read_csv(f, sep=";", usecols=COLONNES, encoding="latin-1")
    for axe, (amont, aval) in ARCS_CIBLES.items():
        sel = brut[(brut["libelle_nd_amont"] == amont) & (brut["libelle_nd_aval"] == aval)]
        morceaux.append(sel.assign(axe=axe))
    print(f"[{i}/{len(fichiers)}] {f.name}", end="\r")

raw = pd.concat(morceaux, ignore_index=True)
print(f"\n{len(raw)} lignes gardées")
raw.groupby(["axe", "iu_ac"]).size().rename("nb_lignes")

63 fichiers pour 2014
[63/63] trafic_capteurs_2014_W51_20141225_20150101.txt
26280 lignes gardées


axe             iu_ac
Champs-Elysees  4264     8760
Convention      5671     8760
Saints-Peres    191      8760
Name: nb_lignes, dtype: int64

## 2. Nettoyage : noms de colonnes, états, dates

D'après la notice de la Ville de Paris (`NOTICE_donnes_trafic_capteurs_permanents`, juin 2019) :

- **`t_1h`** est en **heure de Paris sans fuseau**, et désigne la **fin** de l'heure mesurée
  (« 01:00 » = trafic de 00h à 01h). Vérifié aussi sur les profils horaires (décalage d'1 h l'hiver,
  2 h l'été vs UTC) et sur le recouvrement fin 2025 avec `trafic_3_arcs.csv` (corrélation 1,00 sans
  décalage : même convention). Les heures du changement d'heure (inexistante en mars, ambiguë en
  octobre) sont retirées : elles deviendront des trous.
- **`k`** (occupation, en %) qualifie le trafic : < 15 % fluide, 15-30 % pré-saturé, 30-50 % saturé, ≥ 50 % bloqué.
- **`etat_trafic`** : 0 Inconnu, 1 Fluide, 2 Pré-saturé, 3 Saturé, 4 Bloqué.
- **`etat_barre`** : selon la notice 0 Inconnu, 1 Ouvert, 2 Barré, 3 Invalide. Mais dans ces exports
  seuls 2 et 3 apparaissent, et sur la période commune avec les données récentes le **code 3
  correspond à la fois à « Ouvert » et à « Invalide »**. Seul le code 2 (= Barré) est fiable :
  on garde le code brut dans `etat_arc_code`.

In [4]:
df = raw.rename(columns={
    "iu_ac": "arc_id",
    "t_1h": "datetime",
    "q": "debit",
    "k": "occupation",
    "etat_barre": "etat_arc_code",
})[["axe", "arc_id", "datetime", "debit", "occupation", "etat_trafic", "etat_arc_code"]]

df["etat_trafic"] = df["etat_trafic"].map(ETAT_TRAFIC)
df["datetime"] = pd.to_datetime(df["datetime"]).dt.tz_localize(
    "Europe/Paris", ambiguous="NaT", nonexistent="NaT"
)
print(f"{df['datetime'].isna().sum()} lignes au changement d'heure retirées")

n_avant = len(df.dropna(subset=["datetime"]))
df = (
    df.dropna(subset=["datetime"])
    .drop_duplicates(subset=["axe", "datetime"])
    .sort_values(["axe", "datetime"])
    .reset_index(drop=True)
)
print(f"{n_avant - len(df)} doublons supprimés -> {len(df)} lignes")
df.head()

6 lignes au changement d'heure retirées
0 doublons supprimés -> 26274 lignes


,axe,arc_id,datetime,debit,occupation,etat_trafic,etat_arc_code
0,Champs-Elysees,4264,2014-01-01 01:00:00+01:00,4.0,0.00000,Fluide,3
1,Champs-Elysees,4264,2014-01-01 02:00:00+01:00,21.0,0.10445,Fluide,3
2,Champs-Elysees,4264,2014-01-01 03:00:00+01:00,417.0,6.12667,Fluide,3
3,Champs-Elysees,4264,2014-01-01 04:00:00+01:00,1089.0,14.39890,Fluide,3
4,Champs-Elysees,4264,2014-01-01 05:00:00+01:00,943.0,10.35220,Fluide,3


## 3. Grille horaire complète (une ligne par heure, même s'il n'y a pas de mesure)

In [5]:
grille_utc = pd.date_range(
    df["datetime"].min().tz_convert("UTC"), df["datetime"].max().tz_convert("UTC"), freq="h"
)

complet = []
for (axe, arc_id), g in df.groupby(["axe", "arc_id"]):
    g = (
        g.assign(ligne_absente=False)
        .set_index(g["datetime"].dt.tz_convert("UTC"))
        .drop(columns="datetime")
        .reindex(grille_utc)
    )
    g["axe"], g["arc_id"] = axe, arc_id
    g["ligne_absente"] = g["ligne_absente"].isna()
    complet.append(g)

df = pd.concat(complet).rename_axis("datetime_utc").reset_index()
df["datetime"] = df["datetime_utc"].dt.tz_convert("Europe/Paris")
df = df[["axe", "arc_id", "datetime", "datetime_utc", "debit", "occupation",
         "etat_trafic", "etat_arc_code", "ligne_absente"]]
print(f"Heures attendues par axe : {len(grille_utc)}")

Heures attendues par axe : 8760


## 4. Bilan rapide de l'année

In [6]:
df.groupby("axe").agg(
    debut=("datetime", "min"),
    fin=("datetime", "max"),
    heures=("datetime", "size"),
    lignes_absentes=("ligne_absente", "sum"),
    debit_manquant_pct=("debit", lambda s: round(s.isna().mean() * 100, 1)),
    occupation_manquant_pct=("occupation", lambda s: round(s.isna().mean() * 100, 1)),
    debit_moyen=("debit", "mean"),
    occupation_moyenne=("occupation", "mean"),
).round(1)

C:\Users\Kirat\AppData\Local\Temp\ipykernel_34056\1501025974.py:10: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(1)


,debut,fin,heures,lignes_absentes,debit_manquant_pct,occupation_manquant_pct,debit_moyen,occupation_moyenne
axe,,,,,,,,
Champs-Elysees,2014-01-01 01:00:00+01:00,2015-01-01 00:00:00+01:00,8760,2,1.0,1.6,1468.5,11.3
Convention,2014-01-01 01:00:00+01:00,2015-01-01 00:00:00+01:00,8760,2,2.0,2.0,345.9,6.3
Saints-Peres,2014-01-01 01:00:00+01:00,2015-01-01 00:00:00+01:00,8760,2,0.6,0.6,553.7,9.2


## 5. Sauvegarde

In [7]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out = OUT_DIR / f"trafic_{YEAR}.csv"
df.to_csv(out, index=False)
print(f"Sauvegardé : {out}  ({len(df)} lignes)")

Sauvegardé : data\processed\annees\trafic_2014.csv  (26280 lignes)
